In [74]:

import gmsh
import numpy as np

def distribution_func(size_min, size_max, portal):
    l1, l2 = portal
    x11, y11, x12, y12 = l1
    x21, y21, x22, y22 = l2
    length1 = np.sqrt((x12 - x11) ** 2 + (y12 - y11) ** 2)
    length2 = np.sqrt((x22 - x21) ** 2 + (y22 - y21) ** 2)
    dx1, dy1 = [(x12 - x11) / length1, (y12 - y11) / length1]
    dx2, dy2 = [(x22 - x21) / length2, (y22 - y21) / length2]

    half_distance = (length1 + length2) / 4

    def _base_distribution_func(x):
        return (np.exp(x) - 1) / (np.exp(1) - 1)

    def _total_segment_length(size_min, size_max, num_segs):
        d = 0
        for i in range(num_segs):
            d += _base_distribution_func(i / num_segs) * size_max + size_min
        return d

    num_segs_min = int(half_distance / size_max) - 1
    num_segs_max = int(half_distance / size_min) + 1
    while True:
        num_segs = (num_segs_min + num_segs_max) // 2
        d = _total_segment_length(size_min, size_max, num_segs)
        if _total_segment_length(size_min, size_max, num_segs - 1) <= half_distance <= d:
            num_segs -= 1
            break
        elif _total_segment_length(size_min, size_max, num_segs + 1) >= half_distance >= d:
            break
        elif d > half_distance:
            num_segs_max = num_segs
        else:
            num_segs_min = num_segs
    interp_nodes = [0.0] * (2 * num_segs + 1)
    for i in range(num_segs - 1):
        interp_nodes[i + 1] = interp_nodes[i] + _base_distribution_func(i / num_segs) * size_max + size_min
    interp_nodes[num_segs] = half_distance
    for i in range(num_segs + 1, 2 * num_segs + 1):
        interp_nodes[i] = 2 * half_distance - interp_nodes[2 * num_segs - i]

    p1 = [(0, 0)] * len(interp_nodes)
    p2 = [(0, 0)] * len(interp_nodes)
    for i, d in enumerate(interp_nodes):
        d /= half_distance * 2
        d1 = d * length1
        d2 = d * length2
        p1[i] = (x11 + dx1 * d1, y11 + dy1 * d1)
        p2[i] = (x21 + dx2 * d2, y21 + dy2 * d2)
        
    return p1, p2

def generate_domain_mesh(portals, domain_side_length=10, mesh_size_regular=0.1, mesh_size_min=0.01):
    
    if mesh_size_regular < mesh_size_min:
        raise ValueError("网格尺寸不合理")

    # 🔥 修复2：只创建点和线，不做布尔/同步（避免销毁实体）
    def add_points_to_mesh(points):
        point_tags = []
        line_tags = []
        for x, y in points:
            pt = gmsh.model.occ.addPoint(x, y, 0, mesh_size_regular)
            point_tags.append(pt)
        # 创建线段
        for i in range(len(point_tags)-1):
            ln = gmsh.model.occ.addLine(point_tags[i], point_tags[i+1])
            line_tags.append(ln)
        return point_tags, line_tags
    
    def add_portal_to_mesh(portal):
        p1, p2 = distribution_func(mesh_size_min, mesh_size_regular, portal)
        p1_tags, l1_tags = add_points_to_mesh(p1)
        p2_tags, l2_tags = add_points_to_mesh(p2)
        return p1_tags, p2_tags, l1_tags + l2_tags
    
    # ===================== 核心修复：正确的 Gmsh 执行顺序 =====================
    gmsh.initialize()
    gmsh.model.add("domain_mesh")
    D = domain_side_length
    
    # 1. 创建正方形域
    domain = gmsh.model.occ.addRectangle(-D, -D, 0, 2 * D, 2 * D)
    
    # 2. 创建所有传送门（点+线），保存所有线段
    all_portal_lines = []
    portal_tags = []
    for portal in portals:
        p1_tags, p2_tags, line_tags = add_portal_to_mesh(portal)
        portal_tags.append((p1_tags, p2_tags))
        all_portal_lines.extend([(1, lt) for lt in line_tags])
    print(all_portal_lines)
    
    # 🔥 修复3：批量执行一次布尔运算（不销毁实体）
    if all_portal_lines:
        gmsh.model.occ.fragment([(2, domain)], all_portal_lines)
    
    # 3. 仅一次同步（关键）
    gmsh.model.occ.synchronize()
    
    # 4. 生成网格
    gmsh.option.setNumber("Mesh.CharacteristicLengthMax", mesh_size_regular)
    gmsh.option.setNumber("Mesh.CharacteristicLengthMin", mesh_size_min)
    gmsh.model.mesh.generate(2)

    # 5. 提取节点和单元
    node_tags, node_coords, _ = gmsh.model.mesh.getNodes()
    Nodes = np.array(node_coords).reshape(-1, 3)[:, :2]

    _, elem_tags, node_tags = gmsh.model.mesh.getElements(2, domain)
    Elements = np.array(node_tags[0], dtype=int).reshape(-1, 3) - 1

    # 🔥 修复1：删除错误的 getElements，替换为【正确获取点的网格节点】
    for p1s, p2s in portal_tags:
        for t1, t2 in zip(p1s, p2s):
            # 正确API：获取几何点对应的网格节点（dim=0）
            _, elem_tags, node_tags = gmsh.model.mesh.getElements(0, t1)
            print(f"几何点 {t1} 对应的网格节点: {elem_tags}")

    gmsh.finalize()
    return Nodes, Elements

# ===================== 测试调用（零报错） =====================
if __name__ == "__main__":
    Nodes, Elements = generate_domain_mesh(
        [[(-0.4, -0.6, 0.6, 0.4), (-0.6, -0.4, 0.4, 0.6)]], 
        domain_side_length=3, 
        mesh_size_regular=0.5, 
        mesh_size_min=0.02
    )
    print("网格生成成功！")
    print(f"节点数量: {len(Nodes)}")
    print(f"三角形数量: {len(Elements)}")

[(1, 5), (1, 6), (1, 7), (1, 8), (1, 9), (1, 10), (1, 11), (1, 12), (1, 13), (1, 14), (1, 15), (1, 16), (1, 17), (1, 18), (1, 19), (1, 20)]
几何点 5 对应的网格节点: [array([1], dtype=uint64)]
几何点 6 对应的网格节点: [array([2], dtype=uint64)]
几何点 7 对应的网格节点: [array([3], dtype=uint64)]
几何点 8 对应的网格节点: [array([4], dtype=uint64)]
几何点 9 对应的网格节点: [array([5], dtype=uint64)]
几何点 10 对应的网格节点: [array([6], dtype=uint64)]
几何点 11 对应的网格节点: [array([7], dtype=uint64)]
几何点 12 对应的网格节点: [array([8], dtype=uint64)]
几何点 13 对应的网格节点: [array([9], dtype=uint64)]
网格生成成功！
节点数量: 637
三角形数量: 1224
